# Notebook 01: Setup and Baseline Test

**Agency Calculus Empirical Validation — Paper C**

This notebook:
1. Bootstraps or verifies the AI Economist fork and dependencies
2. Verifies the simulation environment works
3. Runs a short baseline smoke test before full training

Target platforms: Colab GPU runtime `2025.07`, local Python 3.10/3.11\n
\n
> Colab users should first run `bash scripts/colab_bootstrap.sh` from the repo root on runtime\n
> version `2025.07` (Python 3.11). This notebook detects the bootstrap marker and skips the\n
> stale inline install path once the Colab environment is prepared.

## 1. Detect Platform & Install Dependencies

In [1]:
import os, sys
from pathlib import Path

# Detect platform
IN_COLAB = 'google.colab' in sys.modules
IN_KAGGLE = os.path.exists('/kaggle')
LOCAL_MODE = not IN_COLAB and not IN_KAGGLE
IS_WINDOWS = os.name == 'nt'

candidate_roots = [Path.cwd(), Path.cwd().parent, Path('/content/ac-validation')]
REPO_ROOT = next((p for p in candidate_roots if (p / 'src').exists()), None)
BOOTSTRAP_MARKER = None if REPO_ROOT is None else REPO_ROOT / '.colab_bootstrap_complete'
BOOTSTRAP_DONE = bool(BOOTSTRAP_MARKER and BOOTSTRAP_MARKER.exists())

# Local runs assume the environment has already been prepared.
RUN_NOTEBOOK_INSTALLS = (IN_COLAB or IN_KAGGLE) and not BOOTSTRAP_DONE

# RLlib smoke tests are optional and disabled by default.
RUN_RLLIB_SMOKE = os.environ.get('AC_VALIDATION_RUN_RLLIB_SMOKE', '0') == '1'

print(f'Colab: {IN_COLAB}, Kaggle: {IN_KAGGLE}, Local: {LOCAL_MODE}, Windows: {IS_WINDOWS}')
print(f'Python: {sys.version}')
print(f'Bootstrap marker present: {BOOTSTRAP_DONE}')
print(f'Notebook-managed installs: {RUN_NOTEBOOK_INSTALLS}')
print(f'RLlib smoke enabled: {RUN_RLLIB_SMOKE}')


Colab: False, Kaggle: False, Local: True, Windows: True
Python: 3.10.11 (tags/v3.10.11:7d4cc5a, Apr  5 2023, 00:38:17) [MSC v.1929 64 bit (AMD64)]
Notebook-managed installs: False
RLlib smoke enabled: False


In [2]:
# Step 1 of 2: Install packages
# In Colab/Kaggle, this notebook can install its own dependencies.
# For local runs, it assumes the active environment is already prepared.

import sys

if RUN_NOTEBOOK_INSTALLS:
    if sys.version_info >= (3, 12):
        raise SystemExit(
            '\nThis Colab path targets Runtime Version 2025.07 (Python 3.11).\n'
            'Switch Colab to Runtime -> Change runtime type -> Runtime version -> 2025.07,\n'
            'then rerun the bootstrap step.'
        )

    if REPO_ROOT is None:
        raise SystemExit(
            '\nCould not find the repo root from this notebook.\n'
            'Clone the repo to /content/ac-validation, cd into it, and rerun.\n'
        )

    bootstrap_script = REPO_ROOT / 'scripts' / 'colab_bootstrap.sh'
    if not bootstrap_script.exists():
        raise SystemExit(f'Bootstrap script not found: {bootstrap_script}')

    print(f'Running bootstrap: {bootstrap_script}')
    !bash {bootstrap_script}
else:
    if BOOTSTRAP_DONE:
        print('Bootstrap marker detected: skipping notebook-managed installs.')
        print('Using the prepared Colab/local environment as-is.')
    else:
        print('Local mode detected: skipping notebook-managed pip installs.')
        print('Using the currently active Python environment as-is.')


Local mode detected: skipping notebook-managed pip installs.
Using the currently active Python environment as-is.


In [3]:
# Step 2 of 2: Restart runtime (Colab only)
# This cell programmatically restarts the Colab kernel so installed packages become importable.

import sys
if 'google.colab' in sys.modules and RUN_NOTEBOOK_INSTALLS:
    from google.colab import runtime
    runtime.unassign()
else:
    print('No automatic restart performed.')


Local/Kaggle mode: no automatic restart performed.


In [4]:
# Add the repo src directory to sys.path for local or Colab execution
import os, sys
from pathlib import Path

candidate_roots = [Path.cwd(), Path.cwd().parent, Path('/content/ac-validation')]
repo_root = next((p for p in candidate_roots if (p / 'src').exists()), None)
if repo_root is None:
    raise RuntimeError('Could not locate repo root containing src/. Start Jupyter from the repo root or open the notebook from there.')

REPO_SRC = str((repo_root / 'src').resolve())
if REPO_SRC not in sys.path:
    sys.path.insert(0, REPO_SRC)
print(f'src on path: {REPO_SRC}')


src on path: C:\Users\crens\Documents\GitHub\ac-validation\src


## 2. Verify AI Economist Environment

In [5]:
# If you see ModuleNotFoundError here after a notebook-managed install,
# restart the kernel and re-run from the top.
try:
    import ai_economist
    version = getattr(ai_economist, '__version__', 'unknown')
    print(f'ai_economist version : {version}')
except ModuleNotFoundError as e:
    raise SystemExit(
        '\nai_economist not found.\n'
        'Did you restart the kernel after the install cell?\n'
        'Solution: Restart the session and run all cells again.'
    ) from e

from ai_economist.foundation.base.base_env import BaseEnvironment
from ai_economist import foundation
print('foundation import     : OK')

import numpy as np
import ray
print(f'numpy version         : {np.__version__}')
print(f'ray version           : {ray.__version__}')


Inside covid19_components.py: 0 GPUs are available.
No GPUs found! Running the simulation on a CPU.
Inside covid19_env.py: 0 GPUs are available.
No GPUs found! Running the simulation on a CPU.
ai_economist version : unknown
foundation import     : OK
numpy version         : 1.26.4
ray version           : 2.3.0


In [6]:
# Standard AI Economist environment configuration
# 4 worker agents + 1 planner, small map for quick testing

env_config = {
    'scenario_name': 'layout_from_file/simple_wood_and_stone',
    'components': [
        {'Build': {'skill_dist': 'pareto', 'payment_max_skill_multiplier': 3}},
        {'ContinuousDoubleAuction': {'max_num_orders': 5}},
        {'Gather': {}},
    ],
    'env_layout_file': 'quadrant_25x25_20each_30clump.txt',
    'starting_agent_coin': 10,
    'n_agents': 4,
    'world_size': [25, 25],
    'episode_length': 1000,
    'multi_action_mode_agents': False,
    'multi_action_mode_planner': True,
    'flatten_observations': False,
    'flatten_masks': True,
    'allow_observation_scaling': True,
    'mixing_weight_gini_vs_coin': 0.0,  # disable the default mixed planner objective for cleaner baseline checks
}

In [7]:
# Create environment
env = foundation.make_env_instance(**env_config)
print(f'Environment created: {type(env).__name__}')
print(f'n_agents: {env.n_agents}')
print(f'World size: {env.world_size}')

obs = env.reset()
print(f'Observation keys: {list(obs.keys())}')
print(f'Agent 0 obs keys: {list(obs["0"].keys()) if isinstance(obs.get("0"), dict) else "flat"}')

Environment created: LayoutFromFile
n_agents: 4
World size: [25, 25]
Observation keys: ['0', '1', '2', '3', 'p']
Agent 0 obs keys: ['world-map', 'world-idx_map', 'world-loc-row', 'world-loc-col', 'world-inventory-Coin', 'world-inventory-Stone', 'world-inventory-Wood', 'time', 'Build-build_payment', 'Build-build_skill', 'ContinuousDoubleAuction-market_rate-Stone', 'ContinuousDoubleAuction-price_history-Stone', 'ContinuousDoubleAuction-available_asks-Stone', 'ContinuousDoubleAuction-available_bids-Stone', 'ContinuousDoubleAuction-my_asks-Stone', 'ContinuousDoubleAuction-my_bids-Stone', 'ContinuousDoubleAuction-market_rate-Wood', 'ContinuousDoubleAuction-price_history-Wood', 'ContinuousDoubleAuction-available_asks-Wood', 'ContinuousDoubleAuction-available_bids-Wood', 'ContinuousDoubleAuction-my_asks-Wood', 'ContinuousDoubleAuction-my_bids-Wood', 'Gather-bonus_gather_prob', 'action_mask']


In [8]:
# Run a few random steps to confirm env is working
import numpy as np

obs = env.reset()
total_rewards = {k: 0.0 for k in obs.keys()}

for step in range(100):
    actions = {}
    for agent_id in range(env.n_agents):
        agent = env.get_agent(str(agent_id))
        # Single-action workers expect one integer in [0, action_spaces).
        actions[str(agent_id)] = int(np.random.randint(0, agent.action_spaces))
    planner = env.get_agent('p')
    # Multi-action planner expects one integer per action subspace.
    actions['p'] = [int(np.random.randint(0, n)) for n in planner.action_spaces]
    
    obs, rewards, done, info = env.step(actions)
    for k, r in rewards.items():
        total_rewards[k] = total_rewards.get(k, 0) + r

print('100 random steps completed')
print('Cumulative rewards (random policy):')
for k, r in sorted(total_rewards.items()):
    print(f'  Agent {k}: {r:.2f}')

100 random steps completed
Cumulative rewards (random policy):
  Agent 0: -3.96
  Agent 1: -7.14
  Agent 2: 0.00
  Agent 3: 0.71
  Agent p: -4.17


## 3. Verify POLI + Metrics Modules

In [9]:
from ac_rewards import sum_reward, nash_reward, jam_reward, get_reward_fn
from poli_agency import compute_agency, geometric_mean
from metrics import gini_coefficient, compute_episode_metrics, MetricsLogger

# Quick smoke tests
utils = [10.0, 5.0, 8.0, 2.0]

print(f'SUM reward:  {sum_reward(utils):.4f}  (expected 25.0)')
print(f'NASH reward: {nash_reward(utils):.4f}')
print(f'JAM reward:  {jam_reward(utils):.4f}  (expected log(2) = {np.log(2):.4f})')

# JAM with zero utility — should return -1e10
print(f'JAM with zero: {jam_reward([0.0, 5.0, 8.0, 2.0])}')

# POLI
agency = compute_agency(
    coin=50.0, inventory={'wood': 5, 'stone': 3},
    reachable_tiles=30, tradeable_goods=2,
    available_actions=12, income_change=5.0,
    observable_fraction=0.3,
)
print(f'\nPOLI agency score: {agency["agency"]:.4f}')
print(f'  P={agency["prerequisites"]:.3f} O={agency["options"]:.3f} '
      f'L={agency["levers"]:.3f} I={agency["impact"]:.3f} K={agency["knowledge"]:.3f}')

# Gini
equal = [10, 10, 10, 10]
unequal = [1, 1, 1, 97]
print(f'\nGini (equal):   {gini_coefficient(equal):.4f}  (expected 0.0)')
print(f'Gini (unequal): {gini_coefficient(unequal):.4f}')

SUM reward:  25.0000  (expected 25.0)
NASH reward: 6.6847
JAM reward:  0.6931  (expected log(2) = 0.6931)
JAM with zero: -10000000000.0

POLI agency score: 0.0406
  P=0.070 O=0.400 L=0.240 I=0.050 K=0.300

Gini (equal):   0.0000  (expected 0.0)
Gini (unequal): 0.7200


## 4. Short Baseline RLlib Training (100K steps)

This confirms RLlib + AI Economist integration works before starting the full experiment.
Expected runtime: ~5-10 minutes on Colab CPU.

In [10]:
RLLIB_AVAILABLE = False
USE_RLLIB_WRAPPER = False
RLLIB_IMPORT_ERROR = None

if RUN_RLLIB_SMOKE:
    try:
        from ray import tune
        from ray.rllib.algorithms.ppo import PPOConfig, PPO
        from ray.rllib.env.wrappers.multi_agent_env_compatibility import MultiAgentEnvCompatibility
        from ai_economist.training.rllib_wrapper import RLlibEnvWrapper
        print('Using ai_economist RLlibEnvWrapper')
        USE_RLLIB_WRAPPER = True
        RLLIB_AVAILABLE = True
    except Exception as e:
        RLLIB_IMPORT_ERROR = repr(e)
        print(f'Skipping RLlib smoke test: {e}')
else:
    print('Skipping RLlib smoke test by default. Set AC_VALIDATION_RUN_RLLIB_SMOKE=1 before launching Jupyter to enable it.')


Skipping RLlib smoke test by default. Set AC_VALIDATION_RUN_RLLIB_SMOKE=1 before launching Jupyter to enable it.


In [11]:
# Initialize Ray only when running the optional RLlib smoke test
if RLLIB_AVAILABLE:
    if ray.is_initialized():
        ray.shutdown()
    ray.init(ignore_reinit_error=True, num_cpus=2, log_to_driver=False)
    print(f'Ray initialized: {ray.is_initialized()}')
else:
    print('Ray init skipped.')


Ray init skipped.


In [12]:
# Minimal training config for baseline verification
# Full training configs are in notebooks 04-06

if RLLIB_AVAILABLE and USE_RLLIB_WRAPPER:
    trainer_config = {
        'env': RLlibEnvWrapper,
        'env_config': {
            'env_config_dict': env_config,
            'num_envs_per_worker': 1,
        },
        'num_workers': 1,
        'num_gpus': 0,
        'train_batch_size': 4000,
        'rollout_fragment_length': 200,
        'framework': 'torch',
    }

    trainer = PPO(config=trainer_config)

    print('Training for 1 iteration (~4000 steps) to verify...')
    result = trainer.train()
    print('Iteration 1 complete')
    print(f"  episode_reward_mean: {result.get('episode_reward_mean', 'N/A')}")
    print(f"  timesteps_total: {result.get('timesteps_total', 'N/A')}")
    print('\nBaseline RLlib integration: OK')
    trainer.stop()
else:
    if RLLIB_IMPORT_ERROR is not None:
        print(f'RLlib smoke test unavailable: {RLLIB_IMPORT_ERROR}')
    print('Running 10K environment steps (no RLlib)...')
    obs = env.reset()
    for step in range(10000):
        actions = {}
        for agent_id in range(env.n_agents):
            agent = env.get_agent(str(agent_id))
            actions[str(agent_id)] = int(np.random.randint(0, agent.action_spaces))
        planner = env.get_agent('p')
        actions['p'] = [int(np.random.randint(0, n)) for n in planner.action_spaces]
        obs, rewards, done, info = env.step(actions)
        if done.get('__all__', False):
            obs = env.reset()
    print('10K steps completed. Environment: OK')


Running 10K environment steps (no RLlib)...
10K steps completed. Environment: OK


## 5. Summary

- AI Economist: installed and verified
- Environment: creates, resets, and steps correctly
- POLI/Metrics/Rewards modules: imported and smoke-tested
- RLlib smoke test: optional, skipped by default on local runs

**Next:** Notebook 02 - implement and test SUM/NASH/JAM reward functions in the AI Economist planner.


In [13]:
# Cleanup
if 'ray' in globals() and ray.is_initialized():
    ray.shutdown()
print('Setup complete. Ready for notebook 02.')


Setup complete. Ready for notebook 02.
